<a href="https://colab.research.google.com/github/adeeljames/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/adeeljames/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(df.shape, "| declining rate:", df["is_declining_label"].mean().round(3))

(30000, 45) | declining rate: 0.542


Section 1: Method choice and why

For Lane 2 (Refresh / Content Opportunity Scoring), I'm using Logistic Regression
as my primary interpretable baseline model, and Random Forest as my stronger
comparison model.

Why these: this is a ranking/scoring problem built on binary classification
(declining vs not), so classification models that output a probability are the
right fit — that probability becomes the ranking score. Logistic Regression is
simple and readable (matches the "core first" philosophy). Random Forest can
capture non-linear interactions between signals (impressions, staleness,
position, CTR) that a single hand-written rule or linear model can't, which is
exactly the gap the starter pipeline already demonstrated (baseline 0.240 ->
random forest 0.740 on Precision@50).

I am NOT using Gradient Boosting this week to keep the comparison simple and
avoid rewarding complexity without first establishing whether a mid-complexity
model already beats the baseline meaningfully.

Section 2: Split design

Split design: CLIENT-HOLDOUT split, not a random row split. Pages from the same
client can share strong similarities (as I observed in my Week 4 top-10 review,
where all top rows came from one client) — a random split could leak a client's
"style" between train and test. Holding out whole clients gives an honest,
harder test.

In [3]:
from sklearn.model_selection import GroupShuffleSplit

features = ["search_volume", "word_count", "impressions_90d", "sessions_90d",
            "content_age_days", "days_since_last_update", "avg_position",
            "ctr", "engagement_rate", "scroll_rate"]

# keep only features that actually exist in this CSV
features = [f for f in features if f in df.columns]
print("Using features:", features)

X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)
y = df["is_declining_label"].values
groups = df["client_id"].values

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print(f"Train: {len(X_train)} rows, {df['client_id'].iloc[train_idx].nunique()} clients")
print(f"Test:  {len(X_test)} rows, {df['client_id'].iloc[test_idx].nunique()} clients")

# confirm no client overlap
overlap = set(df['client_id'].iloc[train_idx]) & set(df['client_id'].iloc[test_idx])
print("Client overlap between train/test:", len(overlap), "(should be 0)")

Using features: ['search_volume', 'word_count', 'impressions_90d', 'sessions_90d', 'content_age_days', 'days_since_last_update', 'avg_position', 'ctr', 'engagement_rate', 'scroll_rate']
Train: 23837 rows, 25 clients
Test:  6163 rows, 7 clients
Client overlap between train/test: 0 (should be 0)


Section 3: Train + compare vs my baseline

In [4]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

# --- Recompute my Week-4 baseline score on the TEST set only, for a fair comparison ---
test_df = df.iloc[test_idx].copy()
test_df["baseline_score"] = (
    (test_df["days_since_last_update"] >= 180).astype(int) *
    (test_df["impressions_90d"] >= 500).astype(int) *
    test_df["impressions_90d"]
)
baseline_p50 = precision_at_k(test_df["baseline_score"], y_test, min(50, len(y_test)))

# --- Logistic Regression ---
from sklearn.linear_model import LogisticRegression
logreg = LogisticRegression(max_iter=1000, class_weight="balanced")
logreg.fit(X_train, y_train)
logreg_scores = logreg.predict_proba(X_test)[:, 1]
logreg_p50 = precision_at_k(logreg_scores, y_test, min(50, len(y_test)))

# --- Random Forest ---
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=200, max_depth=8, class_weight="balanced", random_state=42)
rf.fit(X_train, y_train)
rf_scores = rf.predict_proba(X_test)[:, 1]
rf_p50 = precision_at_k(rf_scores, y_test, min(50, len(y_test)))

print("Model vs Baseline — Precision@50 (client-holdout test set):")
print(f"  My Week-4 baseline rule:  {baseline_p50:.3f}")
print(f"  Logistic Regression:      {logreg_p50:.3f}")
print(f"  Random Forest:            {rf_p50:.3f}")

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Model vs Baseline — Precision@50 (client-holdout test set):
  My Week-4 baseline rule:  0.620
  Logistic Regression:      0.560
  Random Forest:            0.640


In [5]:
# Also report ROC AUC and Average Precision for a fuller picture
from sklearn.metrics import roc_auc_score, average_precision_score

for name, scores in [("Logistic Regression", logreg_scores), ("Random Forest", rf_scores)]:
    auc = roc_auc_score(y_test, scores)
    ap = average_precision_score(y_test, scores)
    print(f"{name}: ROC AUC={auc:.3f}, Average Precision={ap:.3f}")

Logistic Regression: ROC AUC=0.530, Average Precision=0.520
Random Forest: ROC AUC=0.598, Average Precision=0.587


Result: Random Forest (0.640) beats my Week-4 baseline (0.620) by a small
margin. Logistic Regression (0.560) actually performs WORSE than the baseline
on this client-holdout split.

This is an honest and useful finding, even though the gap is much smaller
than the starter pipeline's famous 3x lift (0.240 -> 0.740). The likely
reason: my Week-4 baseline rule is already fairly strong (0.620), because it
directly encodes two signals I confirmed were real in my Week-4 signal audit
(CTR-vs-position was CONFIRMED). A hand rule built on a genuinely confirmed
signal is a much harder baseline to beat than a naive or untested rule.

Logistic Regression underperforming suggests the relationship between my
features and decline is not well captured by a linear decision boundary —
which is consistent with Random Forest (which can model non-linear
interactions) doing better. This is exactly the kind of result Section 1
predicted: non-linear models should help when signals interact in complex
ways.

Section 4: Errors and interpretation

In [6]:
# Feature importance from Random Forest
importances = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)
print("Random Forest feature importances:")
print(importances)

Random Forest feature importances:
impressions_90d           0.283371
avg_position              0.203355
content_age_days          0.159285
word_count                0.111820
ctr                       0.062729
scroll_rate               0.049264
sessions_90d              0.049123
days_since_last_update    0.044554
search_volume             0.018753
engagement_rate           0.017744
dtype: float64


In [7]:
# Look at a few false positives and false negatives from the Random Forest
test_df["rf_score"] = rf_scores
test_df["actual_label"] = y_test

top50_idx = np.argsort(-rf_scores)[:50]
top50 = test_df.iloc[top50_idx]

false_positives = top50[top50["actual_label"] == 0]
print(f"False positives in top 50: {len(false_positives)}")
false_positives[["impressions_90d", "days_since_last_update", "avg_position", "ctr", "rf_score"]].head(5)

False positives in top 50: 18


,impressions_90d,days_since_last_update,avg_position,ctr,rf_score
11061,1191,103,23.1,0.00,0.809011
10080,1525,103,32.6,0.00,0.808868
28718,264,104,22.2,0.00,0.805750
22524,870,104,17.6,0.11,0.803845
5011,335,104,31.3,0.00,0.803009


Feature Importance

Feature importance (Random Forest):
  impressions_90d           0.283
  avg_position              0.203
  content_age_days          0.159
  word_count                0.112
  ctr                       0.063
  scroll_rate               0.049
  sessions_90d              0.049
  days_since_last_update    0.045
  search_volume             0.019
  engagement_rate           0.018

The top three drivers are impressions_90d (28%), avg_position (20%), and
content_age_days (16%) -- together over 65% of the model's decision-making.
Notably, days_since_last_update (my baseline rule's core signal) ranks only
7th at 4.5% importance -- much lower than I expected. This partly explains
why my Week-4 baseline (built almost entirely around staleness) was already
close to Random Forest's performance: the model found that RAW VISIBILITY
(impressions, position) matters more for predicting decline than staleness
alone, which lines up with my Week-4 Signal A finding that staleness alone
was only a MIXED signal, not a clean predictor.

False positives

False positives in the Random Forest's top 50: 18 out of 50 (36%).

Looking at the actual false-positive rows, a clear pattern jumps out: every
one of them has ctr = 0.00 despite meaningful impressions (264-1,525) and
mid-range average position (17-32). The model is scoring these pages very
high (0.80+) largely on impressions and position, but a CTR of exactly 0
strongly suggests these pages may have a data quality issue (e.g. impressions
without any tracked clicks) rather than genuine declining performance.

This is a useful, actionable error pattern: before trusting the model's top
recommendations, a reviewer should treat ctr = 0.00 rows as needing a data
sanity check first, not an automatic refresh decision -- a page truly
capturing zero clicks from real impressions is unusual enough to warrant
verification before spending reviewer time on it.

Final Closing Paragraph

Overall model quality: ROC AUC and Average Precision confirm the picture from
Precision@50. Random Forest (AUC=0.598, AP=0.587) modestly outperforms
Logistic Regression (AUC=0.530, AP=0.520), but both are much closer to random
(AUC=0.500) than to a strong classifier. This is an honest and important
result: with only 10 available observable features and a proxy label
(trend_direction, calculated from the current window rather than a validated
future outcome), there simply isn't a lot of separable signal for a model to
find in this slice.

What this tells me for next steps: rather than reaching for a more complex
model (e.g. Gradient Boosting) to chase a higher score, the more honest next
move is to improve the LABEL itself -- moving from the current-window proxy
(trend_direction == "down") to a genuine future-window label (e.g. prior 90
days of features -> decline over the next 30 days), as the lane guide
recommends. A weak model on a weak proxy label is expected; the fix is likely
in the target definition, not in adding model complexity.

Bottom line vs my Week-4 baseline: Random Forest (Precision@50 = 0.640) beats
my hand-written rule (Precision@50 = 0.620) by a small but real margin, and
also shows better ranking quality overall (AUC/AP). I'm keeping Random Forest
as my model of choice going into Week 6's validation audit, while noting this
result is modest and the real opportunity for improvement is a stronger label
definition, not a stronger model.